# 配对交易策略分析

基于价差 z-score 的配对交易策略研发与验证。

流程：
1. 静态配对筛选（协整检验）
2. 价差与 z-score 可视化
3. 参数扫描（entry_zscore × exit_zscore × lookback）
4. Walk-forward 验证
5. 与已有策略对比

In [ ]:
import sys

sys.path.insert(0, '..')

import matplotlib
import pandas as pd

matplotlib.use('Agg')
from pathlib import Path

import matplotlib.pyplot as plt

from analysis.param_sweep import best_result, run_sweep
from analysis.plot import plot_sweep_heatmap
from config.loader import load_config
from data.fetcher import fetch_daily
from data.filters import detect_limit_price, detect_suspension
from data.storage import load_parquet, save_parquet
from data.universe import resolve_universe
from factors.cointegration import (
    calc_coint_pvalue,
    calc_half_life,
    calc_spread,
    calc_spread_zscore,
)
from strategies.builtin.pair_trading import pair_trading_signal

print('All imports OK')

## Step 1: 加载数据

In [ ]:
cfg = load_config(Path('../configs/pair_trading.yaml'))
universe_cfg = cfg.get('universe', {})

STOCKS = resolve_universe(universe_cfg)
START = universe_cfg.get('start_date', '2023-01-01')
END = universe_cfg.get('end_date', '2026-05-23')
RAW_DIR = Path('../data/raw')

frames = []
for code in STOCKS:
    path = RAW_DIR / f'{code}.parquet'
    if path.exists():
        df = load_parquet(path)
    else:
        df = fetch_daily(code, START, END)
        save_parquet(df, path)
    frames.append(df)

data = pd.concat(frames, ignore_index=True)
data = detect_limit_price(data)
data = detect_suspension(data)

print(f'Data: {len(data)} rows, {data["code"].nunique()} stocks')
print(f'Date range: {data["date"].min().date()} ~ {data["date"].max().date()}')

## Step 2: 静态配对筛选（协整检验）

对所有候选配对做 Engle-Granger 协整检验，筛选 p-value < 0.05 的配对。

In [ ]:
# 候选配对（同行业）
candidate_pairs = [
    ('600036', '601166'),  # 招行 vs 兴业
    ('600519', '000858'),  # 茅台 vs 五粮液
    ('600519', '000568'),  # 茅台 vs 泸州老窖
    ('000858', '000568'),  # 五粮液 vs 泸州老窖
    ('000333', '000651'),  # 美的 vs 格力
]

# 构建每只股票的 close 价格 DataFrame
close_dfs = {}
for code in STOCKS:
    sub = data[data['code'] == code][['date', 'close']].copy()
    sub = sub.sort_values('date').reset_index(drop=True)
    close_dfs[code] = sub

print('Pair | P-value | Half-life | Status')
print('-' * 55)
selected_pairs = []
for a, b in candidate_pairs:
    if a not in close_dfs or b not in close_dfs:
        print(f'{a}/{b} | N/A | N/A | MISSING DATA')
        continue
    pval = calc_coint_pvalue(close_dfs[a], close_dfs[b], min_obs=60)
    spread = calc_spread(close_dfs[a], close_dfs[b], beta=None)
    hl = calc_half_life(spread, window=60)
    status = 'PASS' if pval < 0.05 else 'FAIL'
    if pval < 0.05:
        selected_pairs.append((a, b))
    print(f'{a}/{b} | {pval:.4f} | {hl:.1f} | {status}')

print(f'\nSelected {len(selected_pairs)} pairs for trading')

## Step 3: 价差与 z-score 可视化

展示选中配对的价差走势和 z-score，标注入场/出场阈值。

In [ ]:
if selected_pairs:
    n_pairs = len(selected_pairs)
    fig, axes = plt.subplots(n_pairs, 2, figsize=(14, 4 * n_pairs))
    if n_pairs == 1:
        axes = axes.reshape(1, -1)

    for i, (a, b) in enumerate(selected_pairs):
        spread = calc_spread(close_dfs[a], close_dfs[b], beta=None)
        zscore = calc_spread_zscore(spread, window=60)

        axes[i, 0].plot(spread.index, spread.values, label='Spread')
        axes[i, 0].set_title(f'{a} vs {b} — Spread')
        axes[i, 0].set_ylabel('Spread')
        axes[i, 0].legend()

        axes[i, 1].plot(zscore.index, zscore.values, label='Z-score', color='orange')
        axes[i, 1].axhline(y=2.0, color='red', linestyle='--', alpha=0.5, label='Entry (2.0)')
        axes[i, 1].axhline(y=-2.0, color='red', linestyle='--', alpha=0.5)
        axes[i, 1].axhline(y=0.5, color='green', linestyle='--', alpha=0.5, label='Exit (0.5)')
        axes[i, 1].axhline(y=-0.5, color='green', linestyle='--', alpha=0.5)
        axes[i, 1].set_title(f'{a} vs {b} — Z-score')
        axes[i, 1].set_ylabel('Z-score')
        axes[i, 1].legend()

    plt.tight_layout()
    output_dir = Path('output')
    output_dir.mkdir(exist_ok=True)
    fig.savefig(output_dir / 'pair_spread_zscore.png', dpi=150, bbox_inches='tight')
    print('Saved: pair_spread_zscore.png')
    plt.close(fig)
else:
    print('No pairs selected — skipping visualization')

## Step 4: 参数扫描

对 entry_zscore × exit_zscore × lookback 做网格搜索。

In [ ]:
# 如果没有通过协整检验的配对，使用配置文件中的默认配对
pairs_to_use = selected_pairs if selected_pairs else [
    ('600036', '601166'),
    ('600519', '000858'),
    ('000333', '000651'),
]

def pair_signal_gen(data, **params):
    return pair_trading_signal(data, pairs=pairs_to_use, **params)

param_grid = {
    'entry_zscore': [1.5, 2.0, 2.5, 3.0],
    'exit_zscore': [0.0, 0.3, 0.5, 0.8],
    'lookback': [30, 60, 90],
}

n_combos = 1
for v in param_grid.values():
    n_combos *= len(v)
print(f'Parameter combinations: {n_combos}')

In [ ]:
results = run_sweep(
    signal_gen=pair_signal_gen,
    param_grid=param_grid,
    data=data,
    capital=1_000_000,
)

print(f'Results: {len(results)} rows')
results.sort_values('sharpe_ratio', ascending=False).head(10)

In [ ]:
output_dir = Path('output')
output_dir.mkdir(exist_ok=True)

# Sharpe 热力图 (entry_zscore vs exit_zscore, lookback=60)
results_60 = results[results['lookback'] == 60]
if not results_60.empty:
    fig = plot_sweep_heatmap(results_60, 'entry_zscore', 'exit_zscore', 'sharpe_ratio',
                             title='Pair Trading — Sharpe (lookback=60)')
    fig.savefig(output_dir / 'pair_sweep_sharpe.png', dpi=150, bbox_inches='tight')
    print('Saved: pair_sweep_sharpe.png')
    plt.close(fig)

# 总收益率热力图
if not results_60.empty:
    fig = plot_sweep_heatmap(results_60, 'entry_zscore', 'exit_zscore', 'total_return',
                             title='Pair Trading — Total Return (lookback=60)')
    fig.savefig(output_dir / 'pair_sweep_return.png', dpi=150, bbox_inches='tight')
    print('Saved: pair_sweep_return.png')
    plt.close(fig)

In [ ]:
best = best_result(results)
print('=== Best Parameters (by Sharpe) ===')
for k in ['entry_zscore', 'exit_zscore', 'lookback']:
    print(f'  {k} = {best[k]}')
print()
print('=== Best Metrics ===')
for k in ['total_return', 'annual_return', 'sharpe_ratio', 'max_drawdown', 'win_rate', 'trade_count']:
    v = best[k]
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

In [ ]:
total = len(results)
profitable = (results['total_return'] > 0).sum()
good_sharpe = (results['sharpe_ratio'] > 0.5).sum()
good_sharpe_low_dd = ((results['sharpe_ratio'] > 0.5) & (results['max_drawdown'] < 0.15)).sum()

print(f'Total combinations: {total}')
print(f'Profitable (return > 0): {profitable} ({profitable/total*100:.1f}%)')
print(f'Good Sharpe (> 0.5): {good_sharpe} ({good_sharpe/total*100:.1f}%)')
print(f'Good Sharpe + Low DD (< 15%): {good_sharpe_low_dd} ({good_sharpe_low_dd/total*100:.1f}%)')
print()
if profitable / total > 0.6:
    print('Result: 参数高原面积较大，策略有一定鲁棒性')
elif profitable / total > 0.3:
    print('Result: 参数敏感性中等，需要 walk-forward 验证')
else:
    print('Result: 参数敏感性高，可能过拟合')

## Step 5: Walk-Forward 验证

按年切分训练/测试窗口，每个窗口重新估计配对和参数，避免前视偏差。

- 训练期：12 个月
- 测试期：3 个月
- 滚动步长：3 个月

In [ ]:
from backtest.engine import BacktestEngine
from portfolio.allocator import equal_weight
from risk.position_limit import apply_position_limit
from risk.tradability import enforce_t1, filter_tradable


def walk_forward_pair_trading(data, pairs, train_months=12, test_months=3):
    """Walk-forward 配对交易验证。"""
    dates = data['date'].sort_values().unique()
    start = pd.Timestamp(dates[0])
    end = pd.Timestamp(dates[-1])

    results = []
    window_start = start

    while window_start + pd.DateOffset(months=train_months + test_months) <= end:
        train_end = window_start + pd.DateOffset(months=train_months)
        test_end = train_end + pd.DateOffset(months=test_months)

        # 训练期：重新估计协整关系
        train_data = data[(data['date'] >= window_start) & (data['date'] < train_end)]
        active_pairs = []
        for a, b in pairs:
            sub_a = train_data[train_data['code'] == a][['date', 'close']].sort_values('date')
            sub_b = train_data[train_data['code'] == b][['date', 'close']].sort_values('date')
            if len(sub_a) < 60 or len(sub_b) < 60:
                continue
            pval = calc_coint_pvalue(sub_a, sub_b, min_obs=60)
            if pval < 0.1:  # 放宽到 0.1 以保留更多配对
                active_pairs.append((a, b))

        if not active_pairs:
            window_start += pd.DateOffset(months=test_months)
            continue

        # 测试期：用训练期参数交易
        test_data = data[(data['date'] >= train_end) & (data['date'] < test_end)]
        if test_data.empty:
            window_start += pd.DateOffset(months=test_months)
            continue

        market = test_data.copy()
        if 'limit_up' not in market.columns:
            market = detect_limit_price(market)
        if 'is_suspended' not in market.columns:
            market = detect_suspension(market)

        signals = pair_trading_signal(
            test_data, pairs=active_pairs,
            entry_zscore=2.0, exit_zscore=0.5, lookback=60
        )
        filtered = filter_tradable(market, signals)
        t1_signals = enforce_t1(filtered)
        prices = test_data[['date', 'code', 'close']]
        positions = equal_weight(t1_signals, prices, capital=1_000_000)
        adjusted = apply_position_limit(positions, max_weight=0.3)

        engine = BacktestEngine(capital=1_000_000)
        result = engine.run(adjusted, prices)

        results.append({
            'period': f'{train_end.date()} ~ {test_end.date()}',
            'pairs_traded': len(active_pairs),
            **result['metrics'],
        })

        window_start += pd.DateOffset(months=test_months)

    return pd.DataFrame(results)

wf_results = walk_forward_pair_trading(data, pairs_to_use)
if not wf_results.empty:
    print('=== Walk-Forward Results ===')
    print(wf_results.to_string(index=False))
    print()
    profitable_periods = (wf_results['total_return'] > 0).sum()
    print(f'Profitable periods: {profitable_periods}/{len(wf_results)}')
else:
    print('No walk-forward results — check data availability')

## Step 6: 结果分析

（运行后填写观察）

- 配对筛选结果
- 参数敏感性
- Walk-forward 表现
- 与已有 4 个失败策略的对比